# InstantID Mobile — Export Pipeline

Notebook chạy toàn bộ pipeline convert InstantID → ONNX → INT8 → OnnxStream cho mobile.

**Runtime:** T4 GPU (Colab free) hoặc A100 (Colab Pro)

**Disk:** Pipeline tự dọn dẹp sau mỗi bước để tránh hết 112 GB.

**Google Drive cache:** Models được cache trên Drive — nếu session bị disconnect, chạy lại từ đầu sẽ restore từ Drive thay vì tải lại ~30 GB.

## Cell 0 — Mount Google Drive + Fix dependencies
Mount Drive trước để cache models. Lần chạy sau sẽ restore từ đây.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Tạo folder cache trên Drive
DRIVE_CACHE = '/content/drive/MyDrive/instantid-mobile-cache'
!mkdir -p {DRIVE_CACHE}/{checkpoints,sdxl-base,sdxl-base-lcm,onnx,onnx-int8,onnxstream-models}

!pip install -q "diffusers>=0.28.0" "huggingface_hub>=0.23.0" "transformers>=4.40.0" onnxscript
import importlib, diffusers
importlib.reload(diffusers)
print(f"diffusers {diffusers.__version__}")

## Cell 1 — Clone repo + setup (skip models nếu đã cache)

In [ ]:
import os
os.chdir('/content')

if not os.path.exists('Instantid-mobile/.git'):
    !git clone https://github.com/Hert4/Instantid-mobile
else:
    print('Repo already cloned')

os.chdir('/content/Instantid-mobile')

# Restore models từ Drive cache (nếu có)
DRIVE_CACHE = '/content/drive/MyDrive/instantid-mobile-cache'

def restore_from_drive(name):
    """Symlink hoặc copy từ Drive cache về local."""
    drive_path = f'{DRIVE_CACHE}/{name}'
    local_path = f'/content/Instantid-mobile/{name}'
    # Kiểm tra Drive cache có data không (ít nhất 1 file > 1MB)
    if os.path.exists(drive_path):
        total = sum(os.path.getsize(os.path.join(dp, f))
                    for dp, _, fns in os.walk(drive_path) for f in fns)
        if total > 1_000_000:  # > 1MB = có data thực
            if not os.path.islink(local_path):
                !rm -rf {local_path}
                !cp -r {drive_path} {local_path}
            print(f'  ✅ Restored {name} from Drive ({total/1e9:.1f} GB)')
            return True
    return False

restored_checkpoints = restore_from_drive('checkpoints')
restored_sdxl = restore_from_drive('sdxl-base')

# Chạy setup — skip models nếu đã restore từ Drive
if restored_checkpoints and restored_sdxl:
    print('\n🚀 Models restored từ Drive — chạy setup --skip-models')
    !./setup.sh --gpu --skip-models
else:
    print('\n📥 Chưa có cache — tải models mới (~30 GB, ~15-30 phút)')
    !./setup.sh --gpu

In [ ]:
# Cache models vừa tải lên Drive (chạy 1 lần, ~10 phút copy)
# Lần sau session mới sẽ restore từ đây thay vì tải lại
DRIVE_CACHE = '/content/drive/MyDrive/instantid-mobile-cache'

import os
def save_to_drive(name):
    src = f'/content/Instantid-mobile/{name}'
    dst = f'{DRIVE_CACHE}/{name}'
    if os.path.exists(src):
        total = sum(os.path.getsize(os.path.join(dp, f))
                    for dp, _, fns in os.walk(src) for f in fns)
        # Chỉ copy nếu Drive chưa có hoặc size khác
        dst_total = 0
        if os.path.exists(dst):
            dst_total = sum(os.path.getsize(os.path.join(dp, f))
                           for dp, _, fns in os.walk(dst) for f in fns)
        if abs(total - dst_total) > 1_000_000:
            print(f'  Saving {name} to Drive ({total/1e9:.1f} GB)...')
            !cp -r {src} {dst}
            print(f'  ✅ Done')
        else:
            print(f'  [skip] {name} already cached on Drive')

save_to_drive('checkpoints')
save_to_drive('sdxl-base')
print('\n✅ Models cached — lần sau restore trong ~1 phút thay vì tải 30 phút')

In [ ]:
# Kiểm tra disk
!df -h /content | tail -1
!echo "---"
!du -sh sdxl-base checkpoints sdxl-base-lcm 2>/dev/null || true

## Cell 2 — Fuse LCM-LoRA

In [ ]:
import os
os.chdir('/content/Instantid-mobile')

DRIVE_CACHE = '/content/drive/MyDrive/instantid-mobile-cache'

# Kiểm tra nếu đã fuse trước đó (restore từ Drive)
if os.path.exists('sdxl-base-lcm/model_index.json'):
    print('✅ sdxl-base-lcm đã tồn tại — skip fuse')
else:
    # Thử restore từ Drive
    drive_lcm = f'{DRIVE_CACHE}/sdxl-base-lcm/model_index.json'
    if os.path.exists(drive_lcm):
        print('🚀 Restoring sdxl-base-lcm từ Drive...')
        !cp -r {DRIVE_CACHE}/sdxl-base-lcm ./sdxl-base-lcm
        print('✅ Restored!')
    else:
        print('Fusing LCM-LoRA (dùng GPU, ~2-3 phút)...')
        !python scripts/fuse_lcm_lora.py --device cuda
        # Cache lên Drive
        print('Caching sdxl-base-lcm lên Drive...')
        !cp -r sdxl-base-lcm {DRIVE_CACHE}/sdxl-base-lcm

assert os.path.exists('sdxl-base-lcm/model_index.json'), 'Fuse failed!'
print('\n✅ Fuse OK!')

In [ ]:
# Xóa SDXL gốc — đã fuse vào sdxl-base-lcm, không cần nữa (~6 GB)
!rm -rf ./sdxl-base
!echo "Freed ~6 GB" && df -h /content | tail -1

## Cell 3 — Export ONNX (từng phần để tránh OOM)

In [ ]:
os.chdir('/content/Instantid-mobile')

# IP-Adapter + ControlNet không cần load full pipeline → export trước
!python scripts/export_all.py --only ip_adapter
!python scripts/export_all.py --only controlnet

In [ ]:
# Text encoders + VAE + UNet (cần load SDXL pipeline)
!python scripts/export_all.py --only text_encoders
!python scripts/export_all.py --only vae
!python scripts/export_all.py --only unet

In [ ]:
# Xóa sdxl-base-lcm — đã export hết sang ONNX
!rm -rf ./sdxl-base-lcm
!echo "Freed sdxl-base-lcm" && df -h /content | tail -1

# Verify ONNX files
!find onnx -name "*.onnx" -exec ls -lh {} \;

## Cell 4 — Quantize INT8

In [ ]:
!python scripts/quantize_all.py

# Xóa ONNX FP16 gốc — đã quantize xong (~9 GB)
!rm -rf ./onnx
!echo "Freed ONNX FP16" && df -h /content | tail -1

## Cell 5 — Convert cho mobile (OnnxStream format)

In [ ]:
!python scripts/convert_for_onnxstream.py

# Kiểm tra kết quả cuối cùng
!du -sh onnxstream-models/
!find onnxstream-models -name "*.yaml" -o -name "*.json" | head -20

## Cell 6 — Lưu kết quả

In [ ]:
# Lưu lên Google Drive (khuyến khích — không mất khi session die)
DRIVE_CACHE = '/content/drive/MyDrive/instantid-mobile-cache'
!cp -r onnxstream-models/ {DRIVE_CACHE}/onnxstream-models/
print('✅ Saved to Google Drive!')
print(f'   Path: {DRIVE_CACHE}/onnxstream-models/')

In [ ]:
# Hoặc tải về máy (zip ~2.5 GB)
import shutil
shutil.make_archive('onnxstream-models', 'zip', 'onnxstream-models')

from google.colab import files
files.download('onnxstream-models.zip')